In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv, find_dotenv

import os 
load_dotenv(find_dotenv(),override=True)

if os.environ["GROQ_API_KEY"]:
    print('API key is found successfuly ')
else:
    print('Please provide an api key ')

API key is found successfuly 


In [ ]:
llm=ChatGroq(model="llama-3.3-70b-versatile")

# **Pydantic Schema**


In [ ]:
from pydantic import BaseModel, Field
from typing import TypedDict,List,Literal

class llm_schema(BaseModel):
    funny_flag: Literal["funny","not_funny"]=Field(...,description="whether this joke is funny or not")
    feedback:str=Field(...,description="Feedback on the Joke ")


llm_with_schema=llm.with_structured_output(llm_schema)

# **Graph Shema**

In [ ]:
class graph_schema(TypedDict):
    topic:str
    joke:str
    funny_flag:str
    feedback:str
    max_iterations:int

# **Node Functions**

In [ ]:
def generate_node(state:graph_schema)-> graph_schema:
    topic=state['topic']
    if state['feedback']:
        response=llm.invoke(f'Please modify the following joke {state['joke']} based on the following feeback: {state['feedback']}')
    else:
        response=llm.invoke(f'Create only one joke about the following topic {topic}')
    
    state['joke']=response.content

    return state

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
def evaluate_node(state:graph_schema)->graph_schema:
    joke=state['joke']
    iteration=state['max_iterations']
    prompt=ChatPromptTemplate(
        [
            ("system","You are a comedy critic , your job is to evaluate the following joke and provide a feedback on how to make it funnier")
            ("human",f"Evaluate the follwoing joke:{joke} , Respond with 'funny' or 'not_funny' and provide feedback if it is not_funny ")
        ]
    )
    chain=prompt | llm_with_schema

    response=chain.invoke({"joke":joke})

    state['funny_flag']=response.funny_flag
    state['feedback']=response.feedback
    state['max_iterations']=iteration + 1
    
    return state